In [1]:
import numpy as np
import pandas as pd
import os
import time
from pathlib import Path

In [2]:
# ======================== 配置区 ========================
BASE_DIR = Path(r"E:\XieYiR\rtl_ml\datasets_validated\datasets_validated")
OUTPUT_DIR = Path(r"E:\XieYiR\rtl_ml")          # 输出目录，按需修改
CHUNK_LEN = 2048
FOLDER_ORDER = [
    "APRS", "FM_broadcast", "FRS_GMRS",
    "ISM_sensors", "NOAA_weather", "noise", "pager"
]
# ========================================================

In [3]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
t0 = time.time()

In [4]:
# ---------- 1. 收集所有 .npy 文件路径（按文件夹顺序 + 文件名排序） ----------
all_files = []
for folder_name in FOLDER_ORDER:
    folder_path = BASE_DIR / folder_name
    if not folder_path.exists():
        print(f"[WARN] 文件夹不存在，已跳过: {folder_path}")
        continue
    npy_files = sorted(folder_path.glob("*.npy"))   # 按文件名排序
    all_files.extend(npy_files)
    print(f"  [{folder_name}] 找到 {len(npy_files)} 个文件")

total_files = len(all_files)
print(f"\n共找到 {total_files} 个 .npy 文件")

  [APRS] 找到 100 个文件
  [FM_broadcast] 找到 200 个文件
  [FRS_GMRS] 找到 100 个文件
  [ISM_sensors] 找到 100 个文件
  [NOAA_weather] 找到 100 个文件
  [noise] 找到 100 个文件
  [pager] 找到 100 个文件

共找到 800 个 .npy 文件


In [5]:
# ---------- 2. 预计算总量 ----------
# 先读第一个文件确认长度
sample_data = np.load(all_files[0], allow_pickle=True).item()
raw_len = len(sample_data["samples"])
chunks_per_file = raw_len // CHUNK_LEN          # 250
discard = raw_len % CHUNK_LEN                    # 0
total_samples = total_files * chunks_per_file

print(f"原始采样点数: {raw_len}")
print(f"每文件切片数: {chunks_per_file}  (丢弃尾部 {discard} 点)")
print(f"总输出样本数: {total_samples}")

原始采样点数: 512000
每文件切片数: 250  (丢弃尾部 0 点)
总输出样本数: 200000


In [6]:
# ---------- 3. 预分配输出数组（complex64 ≈ 3.05 GB） ----------
out_npy_path = OUTPUT_DIR / "dataset_rtl.npy"
print(f"\n正在预分配 dataset_rtl.npy  ({total_samples}×{CHUNK_LEN}, complex64) ...")

# 使用 memmap 避免一次性占用过多内存；如果内存充裕可换成 np.empty
dataset = np.lib.format.open_memmap(
    out_npy_path,
    mode="w+",
    dtype=np.complex64,
    shape=(total_samples, CHUNK_LEN)
)



正在预分配 dataset_rtl.npy  (200000×2048, complex64) ...


In [7]:
# ---------- 4. 逐文件读取 → 切片 → 写入 ----------
label_rows = []          # 收集 CSV 行
global_idx = 0

for fi, npy_path in enumerate(all_files):
    data = np.load(npy_path, allow_pickle=True).item()
    samples = data["samples"]          # (512000,) complex128
    mode    = data["label"]
    snr     = data["snr_db"]

    # 无重叠切片：取前 chunks_per_file*2048 个点
    usable = samples[: chunks_per_file * CHUNK_LEN]          # (512000,)
    chunks = usable.reshape(chunks_per_file, CHUNK_LEN)      # (250, 2048)

    # 写入 memmap（自动转为 complex64）
    dataset[global_idx : global_idx + chunks_per_file] = chunks

    # 记录标签（每个切片的 idx 对应其在 dataset 中的行号）
    for c in range(chunks_per_file):
        label_rows.append((global_idx + c, mode, snr))

    global_idx += chunks_per_file

    # 进度
    if (fi + 1) % 50 == 0 or (fi + 1) == total_files:
        elapsed = time.time() - t0
        print(f"  [{fi+1:>4}/{total_files}]  已写入 {global_idx:>7} 条  "
              f"({elapsed:.1f}s)")

# 刷新 memmap 到磁盘
dataset.flush()
del dataset          # 释放映射

  [  50/800]  已写入   12500 条  (341.6s)
  [ 100/800]  已写入   25000 条  (342.1s)
  [ 150/800]  已写入   37500 条  (342.6s)
  [ 200/800]  已写入   50000 条  (343.1s)
  [ 250/800]  已写入   62500 条  (343.6s)
  [ 300/800]  已写入   75000 条  (344.1s)
  [ 350/800]  已写入   87500 条  (344.7s)
  [ 400/800]  已写入  100000 条  (345.1s)
  [ 450/800]  已写入  112500 条  (345.6s)
  [ 500/800]  已写入  125000 条  (346.1s)
  [ 550/800]  已写入  137500 条  (346.6s)
  [ 600/800]  已写入  150000 条  (347.1s)
  [ 650/800]  已写入  162500 条  (347.6s)
  [ 700/800]  已写入  175000 条  (348.1s)
  [ 750/800]  已写入  187500 条  (348.6s)
  [ 800/800]  已写入  200000 条  (349.1s)


In [8]:
# ---------- 5. 保存 CSV ----------
csv_path = OUTPUT_DIR / "datalabel_rtl.csv"
df = pd.DataFrame(label_rows, columns=["idx", "mode", "snr"])
df.to_csv(csv_path, index=False)

In [9]:
# ---------- 6. 验证 ----------
print("\n========== 验证 ==========")
ds = np.load(out_npy_path, mmap_mode="r")
print(f"dataset_rtl.npy  shape={ds.shape}  dtype={ds.dtype}")
df_check = pd.read_csv(csv_path)
print(f"datalabel_rtl.csv  rows={len(df_check)}  columns={list(df_check.columns)}")
print(df_check.head(3))
print(f"\n总耗时: {time.time()-t0:.1f}s")
print("✅ 转换完成！")


========== 验证 ==========
dataset_rtl.npy  shape=(200000, 2048)  dtype=complex64
datalabel_rtl.csv  rows=200000  columns=['idx', 'mode', 'snr']
   idx  mode       snr
0    0  APRS  6.715727
1    1  APRS  6.715727
2    2  APRS  6.715727

总耗时: 552.2s
✅ 转换完成！
